# Learn CNNs and AlexNet with PyTorch — every code line commented

**Open in Google Colab:** File → Upload notebook. Choose a GPU runtime if available, then run the cells from top to bottom. Colab normally already provides PyTorch, Torchvision, Matplotlib and tqdm; there is no package reinstall cell.

Both examples classify CIFAR-10 colour pictures into airplane, automobile, bird, cat, deer, dog, frog, horse, ship and truck.

- **Small CNN:** random starting weights; all layers learn from the training subset. Input size is 32 × 32.
- **AlexNet:** a pretrained CNN; its existing layers are frozen and only a new 10-class final layer learns. Its official transform produces 224 × 224 inputs. The frozen network stays in eval mode to disable its dropout; eval mode does not disable gradients for the new head.
- Defaults use 10,000 training and 2,000 separate test examples for three epochs. This is a learning run, not a guaranteed accuracy target. The full CIFAR-10 archive is downloaded even when a subset is used.
- Test examples are not used by the training loop. For later hyperparameter experiments, split a validation set out of the training data rather than repeatedly selecting settings on the test set.
- AlexNet's pretrained download is roughly 233 MB. Image enlargement supports the pretrained model; it does not create new visual detail.
- Dataset downloads and training run in your Colab session. This notebook contains no precomputed accuracy or executed training results.
- If DataLoader workers fail in a particular runtime, change both `num_workers=2` settings to `num_workers=0`.

**Important:** AlexNet is itself a CNN. We are comparing a small custom CNN with a particular larger, pretrained CNN. The training regimes differ, so final accuracy is not a fair architecture-only comparison.


## 1. Setup


In [ ]:
import random  # Use Python's random-number generator.
import torch  # Import the main PyTorch library.
from torch import nn  # Import neural-network layers and loss functions.
from torch.utils.data import DataLoader, Subset  # Load examples in batches and select subsets.
from torchvision import datasets, transforms, models  # Import image datasets, processing tools, and models.
import matplotlib.pyplot as plt  # Draw images and learning curves.
from tqdm.auto import tqdm  # Display training progress bars in Colab.

SEED = 42  # Choose a repeatable starting seed; exact GPU results can still vary.
random.seed(SEED)  # Seed Python's random generator.
torch.manual_seed(SEED)  # Seed PyTorch's random generator.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Use a GPU when available.
TRAIN_SIZE = 10000  # Use a smaller training subset for this first learning run; maximum is 50000.
TEST_SIZE = 2000  # Evaluate on a separate test subset; maximum is 10000.
EPOCHS = 3  # Pass through the training subset three times.
print("Device:", device)  # Confirm whether this session is using a GPU or CPU.
print("PyTorch:", torch.__version__)  # Show the installed version for troubleshooting.


## 2. Download images and prepare batches


In [ ]:
raw_train = datasets.CIFAR10(root="./data", train=True, download=True)  # Download training images and labels.
raw_test = datasets.CIFAR10(root="./data", train=False, download=True)  # Download separate test images and labels.
classes = raw_train.classes  # Read the ten category names, such as cat, dog, ship, and truck.
train_indices = torch.randperm(len(raw_train), generator=torch.Generator().manual_seed(SEED))[:TRAIN_SIZE].tolist()  # Select repeatable training IDs.
test_indices = torch.randperm(len(raw_test), generator=torch.Generator().manual_seed(SEED + 1))[:TEST_SIZE].tolist()  # Select repeatable test IDs.
print("Classes:", classes)  # Show the categories both models will predict.

def make_loaders(transform, batch_size):  # Prepare the same example IDs with a model-specific image transform.
    train_data = datasets.CIFAR10(root="./data", train=True, transform=transform)  # Process training images when requested.
    test_data = datasets.CIFAR10(root="./data", train=False, transform=transform)  # Process test images the same way.
    train_subset = Subset(train_data, train_indices)  # Keep only the selected training examples.
    test_subset = Subset(test_data, test_indices)  # Keep only the selected test examples.
    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=device.type == "cuda")  # Shuffle training batches.
    test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=device.type == "cuda")  # Keep testing order fixed.
    return train_loader, test_loader  # Return both batch loaders.


## 3. Shared training, testing, and prediction functions


In [ ]:
def train_model(model, loader, optimizer, epochs, frozen_backbone=False):  # Train either model using the same basic loop.
    loss_function = nn.CrossEntropyLoss()  # Measure classification mistakes from raw scores and integer labels.
    history = []  # Store each epoch's average training loss.
    for epoch in range(epochs):  # Repeat the training process for the chosen number of epochs.
        model.train(not frozen_backbone)  # Train the CNN normally; keep frozen AlexNet dropout disabled.
        total_loss = 0.0  # Reset the accumulated loss for this epoch.
        correct = 0  # Reset the number of correct training predictions.
        seen = 0  # Reset the number of training examples processed.
        for images, labels in tqdm(loader, desc=f"Epoch {epoch + 1}/{epochs}"):  # Read one batch at a time.
            images = images.to(device)  # Move the image batch onto the selected device.
            labels = labels.to(device)  # Move the correct category IDs onto the same device.
            optimizer.zero_grad()  # Clear gradients left over from the previous batch.
            scores = model(images)  # Run the forward pass to produce ten raw scores per image.
            loss = loss_function(scores, labels)  # Compare the raw scores with the correct answers.
            loss.backward()  # Use backpropagation to calculate gradients for trainable parameters.
            optimizer.step()  # Let the optimizer update those trainable parameters.
            total_loss += loss.item() * labels.size(0)  # Add the batch loss weighted by its example count.
            correct += (scores.argmax(dim=1) == labels).sum().item()  # Count correct predictions in this batch.
            seen += labels.size(0)  # Add the number of examples in this batch.
        history.append(total_loss / seen)  # Save the average loss per training example.
        print(f"Training loss: {history[-1]:.4f} | Training accuracy: {100 * correct / seen:.2f}%")  # Report this epoch's results.
    return history  # Return the learning history for plotting.

def test_model(model, loader):  # Measure performance on images that were not used for training.
    model.eval()  # Switch dropout and other relevant layers to evaluation behaviour.
    correct = 0  # Start the correct-prediction counter at zero.
    seen = 0  # Start the example counter at zero.
    with torch.no_grad():  # Disable gradient tracking to save memory during evaluation.
        for images, labels in tqdm(loader, desc="Testing"):  # Read each test batch.
            images = images.to(device)  # Move test images to the selected device.
            labels = labels.to(device)  # Move their correct labels to the same device.
            scores = model(images)  # Calculate ten scores for each test image.
            predicted = scores.argmax(dim=1)  # Pick the category with the highest score.
            correct += (predicted == labels).sum().item()  # Count matching predictions and labels.
            seen += labels.size(0)  # Count the evaluated examples.
    accuracy = 100 * correct / seen  # Convert the fraction of correct answers into a percentage.
    print(f"Test accuracy: {accuracy:.2f}% on {seen} images")  # Display the final test result.
    return accuracy  # Return the percentage for the results table.

def show_predictions(model, loader, title):  # Display eight test predictions beside their correct labels.
    model.eval()  # Use evaluation behaviour while making predictions.
    images, labels = next(iter(loader))  # Read the first ordered test batch.
    with torch.no_grad():  # Avoid tracking gradients for this display.
        predicted = model(images[:8].to(device)).argmax(dim=1).cpu()  # Predict the first eight labels and move them to CPU.
    figure, axes = plt.subplots(2, 4, figsize=(13, 6))  # Create eight image panels.
    for i, axis in enumerate(axes.flat):  # Visit each panel and its matching image index.
        original_image, _ = raw_test[test_indices[i]]  # Retrieve the original image without normalisation or resizing.
        axis.imshow(original_image)  # Draw that original colour image.
        colour = "green" if predicted[i].item() == labels[i].item() else "red"  # Mark correct predictions green and mistakes red.
        axis.set_title(f"Predicted: {classes[predicted[i].item()]}\nActual: {classes[labels[i].item()]}", color=colour)  # Show both labels.
        axis.axis("off")  # Hide chart axes around the picture.
    figure.suptitle(title)  # Add the model name above the image grid.
    plt.tight_layout()  # Arrange the panels to reduce overlap.
    plt.show()  # Display the prediction grid in Colab.


## 4. Model A: build and train a small CNN from scratch


In [ ]:
cnn_transform = transforms.Compose([  # Define the image processing used by our small CNN.
    transforms.ToTensor(),  # Convert an RGB image to a [3, 32, 32] tensor with values from 0 to 1.
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # Rescale each colour channel to approximately -1 through 1.
])  # Finish the CNN image-processing pipeline.
cnn_train_loader, cnn_test_loader = make_loaders(cnn_transform, batch_size=64)  # Prepare batches of up to 64 images.

cnn = nn.Sequential(  # Build a model by passing each layer's output into the next layer.
    nn.Conv2d(3, 16, kernel_size=3, padding=1),  # Turn RGB input into 16 feature maps: [batch, 16, 32, 32].
    nn.ReLU(),  # Replace negative responses with zero without changing the shape.
    nn.MaxPool2d(kernel_size=2, stride=2),  # Shrink each map: [batch, 16, 16, 16].
    nn.Conv2d(16, 32, kernel_size=3, padding=1),  # Combine all 16 incoming maps into 32 new maps: [batch, 32, 16, 16].
    nn.ReLU(),  # Add another nonlinear activation without changing the shape.
    nn.MaxPool2d(kernel_size=2, stride=2),  # Shrink the maps again: [batch, 32, 8, 8].
    nn.Flatten(),  # Arrange 32 times 8 times 8 feature values into a list: [batch, 2048].
    nn.Linear(32 * 8 * 8, 64),  # Connect all 2048 features to 64 hidden neurons: [batch, 64].
    nn.ReLU(),  # Apply an activation to the hidden neurons.
    nn.Linear(64, 10),  # Produce one raw score for each CIFAR-10 category: [batch, 10].
).to(device)  # Move the model's parameters to the selected device.
print(cnn)  # Display the CNN's layers.
print("Trainable CNN parameters:", sum(p.numel() for p in cnn.parameters() if p.requires_grad))  # Count its adjustable numbers.
cnn_optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001)  # Let Adam update every CNN weight and bias.
cnn_history = train_model(cnn, cnn_train_loader, cnn_optimizer, EPOCHS)  # Learn from the training examples.
cnn_accuracy = test_model(cnn, cnn_test_loader)  # Evaluate the trained CNN on the separate test subset.
show_predictions(cnn, cnn_test_loader, "Small CNN: test predictions")  # Show correct predictions and mistakes.


## 5. Model B: adapt pretrained AlexNet


In [ ]:
alex_weights = models.AlexNet_Weights.DEFAULT  # Select Torchvision's pretrained ImageNet weights.
alex_transform = alex_weights.transforms()  # Apply their expected resize, 224-by-224 crop, and colour normalisation.
alex_train_loader, alex_test_loader = make_loaders(alex_transform, batch_size=32)  # Use smaller batches because images are now larger.
alex = models.alexnet(weights=alex_weights)  # Download and load pretrained AlexNet, including its five convolution layers.
for parameter in alex.parameters():  # Visit every existing AlexNet parameter.
    parameter.requires_grad = False  # Freeze the pretrained parameters so training will not update them.
alex.classifier[6] = nn.Linear(alex.classifier[6].in_features, 10)  # Replace the 1000-class final layer with a new trainable 10-class layer.
alex = alex.to(device)  # Move AlexNet and its new output layer to the selected device.
print(alex)  # Display all layers, including the convolution and classifier sections.
print("Trainable AlexNet parameters:", sum(p.numel() for p in alex.parameters() if p.requires_grad))  # Count only the new output layer's parameters.
alex_optimizer = torch.optim.Adam(alex.classifier[6].parameters(), lr=0.001)  # Update only the new final layer.
alex_history = train_model(alex, alex_train_loader, alex_optimizer, EPOCHS, frozen_backbone=True)  # Train the final layer with the pretrained network kept in eval mode.
alex_accuracy = test_model(alex, alex_test_loader)  # Evaluate AlexNet on the same test image IDs used for the CNN.
show_predictions(alex, alex_test_loader, "AlexNet transfer learning: test predictions")  # Show its predictions on the same eight pictures.


## 6. Inspect layer shapes, plot learning, and save weights


In [ ]:
cnn.eval()  # Use evaluation behaviour for the shape demonstration.
sample = torch.zeros(1, 3, 32, 32, device=device)  # Create one dummy RGB image solely to inspect tensor shapes.
with torch.no_grad():  # Inspect shapes without building a gradient-tracking graph.
    for layer in cnn:  # Visit the CNN's layers in their execution order.
        sample = layer(sample)  # Pass the current tensor through this layer.
        print(type(layer).__name__, "->", tuple(sample.shape))  # Display how the layer changes its shape.

plt.figure(figsize=(8, 4))  # Create a chart of the training loss.
plt.plot(range(1, EPOCHS + 1), cnn_history, marker="o", label="Small CNN")  # Plot CNN training loss after each epoch.
plt.plot(range(1, EPOCHS + 1), alex_history, marker="o", label="Pretrained AlexNet + new head")  # Plot AlexNet training loss.
plt.xlabel("Epoch")  # Label the horizontal axis.
plt.ylabel("Average training loss")  # Label the vertical axis.
plt.legend()  # Identify the two curves.
plt.grid(alpha=0.3)  # Add a faint grid to help read the values.
plt.show()  # Display the learning curves.
print(f"Small CNN test accuracy: {cnn_accuracy:.2f}%")  # Report the CNN's final test accuracy.
print(f"AlexNet transfer-learning test accuracy: {alex_accuracy:.2f}%")  # Report AlexNet's final test accuracy.
print("These demonstrate different training approaches; this is not a controlled architecture comparison.")  # Explain the pretraining and input-size differences.
torch.save({"model_state": cnn.state_dict(), "classes": classes}, "small_cnn_cifar10.pth")  # Save all CNN weights and the class names in this Colab session.
torch.save({"head_state": alex.classifier[6].state_dict(), "classes": classes, "base_weights": "AlexNet_Weights.IMAGENET1K_V1"}, "alexnet_cifar10_head.pth")  # Save only the new AlexNet head; reload it onto the same pretrained base later.
print("Saved small_cnn_cifar10.pth and alexnet_cifar10_head.pth in the Colab Files panel.")  # Explain where to download the saved weights.


## Official references
- [PyTorch transfer learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [Torchvision AlexNet and its expected preprocessing](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html)
- [PyTorch image-classification tutorial](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)

Validation during preparation: all six code cells were syntax-checked, and every nonblank code line was checked for a comment. Training has not been executed during preparation.
